In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251230 PCDS Changes - AGRIWeather Round 3.xlsx', header = 0)   # pandas automatically uses openpyxl
id_issues = [
    "ID",
    "ID, Name",
]

name_issues = [
    "Name",
    "ID, Name",
]

df_id = df[
    df["ISSUE"].str.strip().isin(id_issues)
]

df_name = df[
    df["ISSUE"].str.strip().isin(name_issues)
]


df_id

,ISSUE,network_id,native_id,station_name,min_obs_time,max_obs_time,lat,lon,elev,station_id
0,ID,10,AGAL001->ALCALILK,Alkali Lake,NaT,NaT,51.790278,-121.251111,677.000,14844.0
1,ID,10,AGBCPRDNCK->BCPRDNCK,BCGPA Dawson Creek,NaT,NaT,55.772100,-120.143000,673.000,14845.0
2,ID,10,AGBCPRBFLT->BCPRBFLT,Bearflats,NaT,NaT,56.246100,-121.308000,504.000,14846.0
3,ID,10,AGOKNGBVST->OKNGBVST,Bella Vista,NaT,NaT,50.255100,-119.341300,437.000,14847.0
4,ID,10,AGOKNGBNVN->OKNGBNVN,Benvoulin,NaT,NaT,49.872000,-119.449100,357.000,14889.0
...,...,...,...,...,...,...,...,...,...,...
72,"ID, Name",10,AGOKNGBLGO->OKGBLGO,Belgo 2->Belgo,NaT,NaT,49.876700,-119.384000,468.000,14888.0
73,"ID, Name",10,di107->DIAMONDS,Diamond S->Diamond South,2007-09-14,2010-09-14,50.851930,-121.867490,450.000,1514.0
74,"ID, Name",10,AGCRESRIVW->CRESRIVW,North Lister (Riverview)->North Lister/Riverview,NaT,NaT,49.061800,-116.508100,609.000,14870.0
75,"ID, Name",10,si107->SLVRHILL,Silver Hills Ranch->Silver Hills,2008-04-17,2010-09-14,50.253333,-118.690556,495.000,1509.0


In [5]:
df_id

df_new = df_id[['native_id', 'station_id']].copy()

# Split on '->'
split_ids = df_new['native_id'].astype(str).str.split('->', expand=True)
df_new['old_native_id'] = split_ids[0].str.strip()
df_new['new_native_id'] = split_ids[1].str.strip()

df_new = df_new.drop(columns=['native_id'])
df_new["station_id"] = pd.to_numeric(df_new["station_id"], errors="coerce").astype("Int64")

df_new

,station_id,old_native_id,new_native_id
0,14844,AGAL001,ALCALILK
1,14845,AGBCPRDNCK,BCPRDNCK
2,14846,AGBCPRBFLT,BCPRBFLT
3,14847,AGOKNGBVST,OKNGBVST
4,14889,AGOKNGBNVN,OKNGBNVN
...,...,...,...
72,14888,AGOKNGBLGO,OKGBLGO
73,1514,di107,DIAMONDS
74,14870,AGCRESRIVW,CRESRIVW
75,1509,si107,SLVRHILL


### In the db

In [6]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### Update the ID

In [8]:

check_sql = sa.text("""
SELECT station_id, native_id
FROM meta_station
WHERE station_id = :station_id
""")

with engine.connect() as conn:
    for _, row in df_new.iterrows():
        res = conn.execute(
            check_sql,
            {"station_id": int(row["station_id"])}
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['station_id']} not found")
            continue

        db_native_id = res.native_id

        if db_native_id == row["new_native_id"]:
            print(
                f"✅ OK to update station {res.station_id}: "
                f"{db_native_id} → {row['new_native_id']}"
            )
        else:
            print(
                f"⚠️ Mismatch for station {res.station_id}: "
                f"DB={db_native_id}, expected={row['new_native_id']}"
            )

✅ OK to update station 14844: ALCALILK → ALCALILK
✅ OK to update station 14845: BCPRDNCK → BCPRDNCK
✅ OK to update station 14846: BCPRBFLT → BCPRBFLT
✅ OK to update station 14847: OKNGBVST → OKNGBVST
✅ OK to update station 14889: OKNGBNVN → OKNGBNVN
✅ OK to update station 1510: BP002 → BP002
✅ OK to update station 14891: BB003 → BB003
✅ OK to update station 14848: BCPRBUIK → BCPRBUIK
✅ OK to update station 14849: CRESCNYN → CRESCNYN
✅ OK to update station 14892: OKNGCWSTB → OKNGCWSTB
✅ OK to update station 1511: CHASESTN → CHASESTN
✅ OK to update station 1512: COLDWATR → COLDWATR
✅ OK to update station 1513: DEEPCREE → DEEPCREE
✅ OK to update station 14894: BCPRDORV → BCPRDORV
✅ OK to update station 1515: DOUGLAKE → DOUGLAKE
✅ OK to update station 14895: OKNGEKLN → OKNGEKLN
✅ OK to update station 14896: OKNGELSN → OKNGELSN
✅ OK to update station 14853: BCPRFMTN → BCPRFMTN
✅ OK to update station 14897: BCPRFLHT → BCPRFLHT
✅ OK to update station 14854: BCPRFRCK → BCPRFRCK
✅ OK to update 

In [9]:
update_station_sql = sa.text("""
UPDATE meta_station
SET native_id = :new_native_id
WHERE station_id = :station_id
  AND native_id = :old_native_id
""")

with engine.begin() as conn:
    for _, row in df_new.iterrows():
        result = conn.execute(
            update_station_sql,
            {
                "station_id": int(row["station_id"]),
                "old_native_id": row["old_native_id"],
                "new_native_id": row["new_native_id"],
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {row['station_id']}: "
                f"{row['old_native_id']} → {row['new_native_id']}"
            )
        else:
            # check why it failed
            res = conn.execute(
                check_sql,
                {"station_id": int(row["station_id"])}
            ).fetchone()

            if res is None:
                print(f"❌ Station {row['station_id']} not found")
            else:
                print(
                    f"⚠️ Skipped station {row['station_id']}: "
                    f"DB={res.native_id}, expected={row['old_native_id']}"
                )

⚠️ Skipped station 14844: DB=ALCALILK, expected=AGAL001
⚠️ Skipped station 14845: DB=BCPRDNCK, expected=AGBCPRDNCK
⚠️ Skipped station 14846: DB=BCPRBFLT, expected=AGBCPRBFLT
⚠️ Skipped station 14847: DB=OKNGBVST, expected=AGOKNGBVST
⚠️ Skipped station 14889: DB=OKNGBNVN, expected=AGOKNGBNVN
⚠️ Skipped station 1510: DB=BP002, expected=bo107
⚠️ Skipped station 14891: DB=BB003, expected=AGBB003
⚠️ Skipped station 14848: DB=BCPRBUIK, expected=AGBCPRBUIK
⚠️ Skipped station 14849: DB=CRESCNYN, expected=AGCRESCNYN
⚠️ Skipped station 14892: DB=OKNGCWSTB, expected=AGOKNGCWSTB
⚠️ Skipped station 1511: DB=CHASESTN, expected=ch107
⚠️ Skipped station 1512: DB=COLDWATR, expected=co107
⚠️ Skipped station 1513: DB=DEEPCREE, expected=de107
⚠️ Skipped station 14894: DB=BCPRDORV, expected=AGBCPRDORV
⚠️ Skipped station 1515: DB=DOUGLAKE, expected=do107
⚠️ Skipped station 14895: DB=OKNGEKLN, expected=AGOKNGEKLN
⚠️ Skipped station 14896: DB=OKNGELSN, expected=AGOKNGELSN
⚠️ Skipped station 14853: DB=BCPRFMTN

In [10]:
verify_sql = sa.text("""
SELECT native_id
FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_new.iterrows():
        # verify
        res = conn.execute(
            verify_sql,
            {"station_id": int(row["station_id"])}
        ).fetchone()

        if res and res.native_id == row["new_native_id"]:
            print(
                f"✅ Verified: station {row['station_id']} "
                f"native_id = {res.native_id}"
            )
        else:
            print(
                f"⚠️ Verification failed for station {row['station_id']}: "
                f"DB={res.native_id if res else None}, "
                f"expected={row['new_native_id']}"
            )

✅ Verified: station 14844 native_id = ALCALILK
✅ Verified: station 14845 native_id = BCPRDNCK
✅ Verified: station 14846 native_id = BCPRBFLT
✅ Verified: station 14847 native_id = OKNGBVST
✅ Verified: station 14889 native_id = OKNGBNVN
✅ Verified: station 1510 native_id = BP002
✅ Verified: station 14891 native_id = BB003
✅ Verified: station 14848 native_id = BCPRBUIK
✅ Verified: station 14849 native_id = CRESCNYN
✅ Verified: station 14892 native_id = OKNGCWSTB
✅ Verified: station 1511 native_id = CHASESTN
✅ Verified: station 1512 native_id = COLDWATR
✅ Verified: station 1513 native_id = DEEPCREE
✅ Verified: station 14894 native_id = BCPRDORV
✅ Verified: station 1515 native_id = DOUGLAKE
✅ Verified: station 14895 native_id = OKNGEKLN
✅ Verified: station 14896 native_id = OKNGELSN
✅ Verified: station 14853 native_id = BCPRFMTN
✅ Verified: station 14897 native_id = BCPRFLHT
✅ Verified: station 14854 native_id = BCPRFRCK
✅ Verified: station 14898 native_id = OKNGGLMR
✅ Verified: station 148

## Update the name

In [11]:
df_name

df_name_new =  df_name[['station_name', 'station_id']].reset_index(drop=True)


def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    
df_name = df_name_new.copy()

df_name[['old_name', 'new_name', 'n_names']] = (
    df_name['station_name'].apply(split_station_name)
)

df_name

,station_name,station_id,old_name,new_name,n_names
0,Belgo 2->Belgo,14888.0,Belgo 2,Belgo,2
1,Diamond S->Diamond South,1514.0,Diamond S,Diamond South,2
2,North Lister (Riverview)->North Lister/Riverview,14870.0,North Lister (Riverview),North Lister/Riverview,2
3,Silver Hills Ranch->Silver Hills,1509.0,Silver Hills Ranch,Silver Hills,2
4,Sunshine Valley->Lower Nicola (Sunshine Valley),1522.0,Sunshine Valley,Lower Nicola (Sunshine Valley),2
5,Belgo->Belgo (Old),1552.0,Belgo,Belgo (Old),2
6,Silver Creek ->Silver Creek (Old),1506.0,Silver Creek,Silver Creek (Old),2


In [ ]:
check_sql = sa.text("""
SELECT
    station_id,
    station_name
FROM meta_history
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        station_id = int(row["station_id"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != old_name:
            print(
                f"⚠️ Station {station_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {old_name}"
            )
        else:
            print(
                f"✅ Station {station_id}, {db_row.station_name} the names match "
            )


✅ Station 14888, Belgo the names match 
✅ Station 1514, Diamond South the names match 
✅ Station 14870, North Lister/Riverview the names match 
✅ Station 1509, Silver Hills the names match 
✅ Station 1522, Lower Nicola (Sunshine Valley) the names match 
✅ Station 1552, Belgo (Old) the names match 
✅ Station 1506, Silver Creek (Old) the names match 


In [12]:
update_sql = sa.text("""
UPDATE meta_history
SET
    station_name  = :new_name
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        station_id = int(row["station_id"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_name": new_name,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_name}) → "
                f"({new_name})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")

✅ Updated station 14888: (Belgo 2) → (Belgo)
✅ Updated station 1514: (Diamond S) → (Diamond South)
✅ Updated station 14870: (North Lister (Riverview)) → (North Lister/Riverview)
✅ Updated station 1509: (Silver Hills Ranch) → (Silver Hills)
✅ Updated station 1522: (Sunshine Valley) → (Lower Nicola (Sunshine Valley))
✅ Updated station 1552: (Belgo) → (Belgo (Old))
✅ Updated station 1506: (Silver Creek) → (Silver Creek (Old))


In [14]:
df_name

,station_name,station_id,old_name,new_name,n_names
0,Belgo 2->Belgo,14888.0,Belgo 2,Belgo,2
1,Diamond S->Diamond South,1514.0,Diamond S,Diamond South,2
2,North Lister (Riverview)->North Lister/Riverview,14870.0,North Lister (Riverview),North Lister/Riverview,2
3,Silver Hills Ranch->Silver Hills,1509.0,Silver Hills Ranch,Silver Hills,2
4,Sunshine Valley->Lower Nicola (Sunshine Valley),1522.0,Sunshine Valley,Lower Nicola (Sunshine Valley),2
5,Belgo->Belgo (Old),1552.0,Belgo,Belgo (Old),2
6,Silver Creek ->Silver Creek (Old),1506.0,Silver Creek,Silver Creek (Old),2


In [18]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_name.iterrows():

        station_id = int(row["station_id"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != new_name:
            print(
                f"⚠️ Station {station_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {new_name}"
            )
        else:
            print(
                f"✅ Station {station_id}, {db_row.station_name} the names match "
            )


✅ Station 14888, Belgo the names match 
✅ Station 1514, Diamond South the names match 
✅ Station 14870, North Lister/Riverview the names match 
✅ Station 1509, Silver Hills the names match 
✅ Station 1522, Lower Nicola (Sunshine Valley) the names match 
✅ Station 1552, Belgo (Old) the names match 
✅ Station 1506, Silver Creek (Old) the names match 


### New data insert

In [19]:
new_issues = [
    "New"
]

df_new = df[
    df["ISSUE"].str.strip().isin(new_issues)
]

df_new

,ISSUE,network_id,native_id,station_name,min_obs_time,max_obs_time,lat,lon,elev,station_id
79,New,10,BCPRBSNC,Bison Creek,NaT,NaT,56.1737,-120.3743,655.0,NaN
80,New,10,CHILEAST,Chilliwack East,NaT,NaT,49.1865,-121.8564,11.0,NaN
81,New,10,BCPRCLAY,Clayhurst,NaT,NaT,56.1890,-120.0667,641.0,NaN
82,New,10,BCVHCTNT,Cottontail,NaT,NaT,54.0384,-123.7983,751.0,NaN
83,New,10,BCVHEDCK,Eden Creek,NaT,NaT,54.1079,-124.1615,724.0,NaN
84,New,10,BCVHFLNS,Fraser Lake North Shore,NaT,NaT,54.0801,-124.8464,677.0,NaN
85,New,10,OKNGGHDS,Giants Head South,NaT,NaT,49.5746,-119.6576,510.0,NaN
86,New,10,GFNURSRY,Grand Forks Nursery,NaT,NaT,49.0071,-118.4820,536.0,NaN
87,New,10,OKNGLMBY,Lumby,NaT,NaT,50.2237,-119.0261,523.0,NaN
88,New,10,NUKKOLAK,Nukko Lake,NaT,NaT,54.0891,-122.9754,772.0,NaN


In [20]:
from sqlalchemy import func

stations_created = []

# for _, row in df_new.iloc[0:2].iterrows():
for _, row in df_new.iterrows():
    
    name = row['station_name']
    nid  = row['native_id']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=row['network_id'])


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['elev'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('Bison Creek', 14972), ('Chilliwack East', 14973), ('Clayhurst', 14974), ('Cottontail', 14975), ('Eden Creek', 14976), ('Fraser Lake North Shore', 14977), ('Giants Head South', 14978), ('Grand Forks Nursery', 14979), ('Lumby', 14980), ('Nukko Lake', 14981), ('Sumas Prairie', 14982), ('Oliver (Panorama)', 14983), ('Summerland Upper South', 14984), ('Trinity Valley', 14985), ('Tappen', 14986), ('Vanderhoof Northside', 14987), ('Vernon BX', 14988), ('West Kelowna', 14989), ('West Kelowna Nursery', 14990)]
